# Thelerine 2.0 — warp-based VTON training on Colab

Two stages, run in order:

1. **Warp** — learns where the garment goes (TPS + residual flow). ~10 min on a T4.
2. **Compose** — learns the copy/synthesise blend. ~30 min on a T4.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. Without a GPU this
notebook still runs, but stage 2 takes many hours.

Checkpoints are written straight to Drive and every cell resumes automatically, so
if Colab disconnects just re-run the cells from the top — nothing is lost.

## 1. Check the GPU

In [ ]:
import torch

print('torch', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')

## 2. Mount Drive and clone the repo

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/kmafatshe/Thelerine2.0_vton-warp.git'
REPO_DIR = '/content/Thelerine2.0_vton-warp'

import os, subprocess

if os.path.isdir(REPO_DIR):
    # Already cloned in this session — pull the latest instead.
    print(subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                         capture_output=True, text=True).stdout)
else:
    print(subprocess.run(['git', 'clone', REPO_URL, REPO_DIR],
                         capture_output=True, text=True).stderr)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 3. Find the dataset

The code expects `person/`, `garments/`, `cihp/` and `segmentation/` under the
dataset root. Your folders may be named differently (the previous project used
`cond/` and `seg/`), so this cell inspects what is actually there and picks the
matching names. Check the printed mapping before moving on.

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/thelerine_ai_data')
OUTPUT_ROOT = Path('/content/drive/MyDrive/thelerine_ai_outputs/vton_warp')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

assert DATA_ROOT.exists(), f'not found: {DATA_ROOT}'

print('Top level of', DATA_ROOT)
for entry in sorted(DATA_ROOT.iterdir()):
    if entry.is_dir():
        n = sum(1 for _ in entry.rglob('*') if _.is_file())
        kids = sorted(c.name for c in entry.iterdir() if c.is_dir())
        print(f'  {entry.name:<20} {n:>5} files   subfolders: {kids}')
    else:
        print(f'  {entry.name}')

In [ ]:
# Map the four roles onto whatever the folders are actually called.
CANDIDATES = {
    'person_dir': ['person', 'people', 'persons', 'model'],
    'garment_dir': ['garments', 'garment', 'cloth', 'clothes'],
    'cihp_dir': ['cihp', 'cond', 'parse', 'parsing', 'label'],
    'segmentation_dir': ['segmentation', 'seg', 'mask', 'masks'],
}

present = {d.name.lower(): d.name for d in DATA_ROOT.iterdir() if d.is_dir()}
DIRS = {}
for role, options in CANDIDATES.items():
    match = next((present[o] for o in options if o in present), None)
    DIRS[role] = match
    print(f'{role:<18} -> {match}')

missing = [r for r in ('person_dir', 'garment_dir') if DIRS[r] is None]
assert not missing, f'could not find {missing}; set DIRS manually in this cell'
if DIRS['cihp_dir'] is None:
    print('\n!! No parse folder found. The agnostic representation needs one —\n'
          '   set DIRS["cihp_dir"] by hand before continuing.')

# Turn the mapping into command-line overrides for the training scripts.
DATA_ARGS = [f'data.root={DATA_ROOT}'] + [
    f'data.{role}={name}' for role, name in DIRS.items() if name
]
print('\noverrides:', ' '.join(DATA_ARGS))

## 4. Sanity-check the data — do not skip this

One mismatched parse map is a whole percent of a small dataset. Look at the
contact sheet below, specifically the **`agnostic` column: the original garment
must be completely gone.** If any of it survives, the model will learn to copy it
and will fall apart on a new garment.

In [ ]:
!python scripts/check_dataset.py --root "{DATA_ROOT}" --samples 6 --out /content/dataset_check.png

In [ ]:
from IPython.display import Image, display

display(Image('/content/dataset_check.png'))

**If `segmentation/` turned out to hold a person silhouette or a garment mask in
*person* space** (the check prints its best guess), add
`data.segmentation_role=ignore` to `DATA_ARGS` below. The flat garment's mask is
then derived by thresholding the product shot's background instead.

## 5. Stage 1 — train the warper

Watch `warp/shape` fall. It is the silhouette agreement between the warped
garment and the region it has to fill, and it is the number that tells you
whether the geometry is being learned.

In [ ]:
WARP_OUT = OUTPUT_ROOT / 'warp'

args = ' '.join(DATA_ARGS)
!python train_warp.py --config configs/warp.yaml {args} \
    output_dir="{WARP_OUT}" \
    train.steps=12000 train.batch_size=8 train.num_workers=2 \
    train.sample_every=1000 train.save_every=1000

In [ ]:
# Newest warp sample sheet. Columns: garment | agnostic | coarse | warped |
# overlay | target | ground truth | flow.
# The `overlay` column is the one that matters — the garment should sit on the
# body in the right place and shape before you start stage 2.
sheets = sorted((WARP_OUT / 'samples').glob('*.png'))
display(Image(str(sheets[-1])))

## 6. Stage 2 — train the composer

The warper is frozen here. Watch the `alpha` column in the samples: it should be
bright and crisp over the garment, meaning the model is *copying* real garment
pixels rather than hallucinating them.

In [ ]:
TRYON_OUT = OUTPUT_ROOT / 'tryon'

!python train_tryon.py --config configs/tryon.yaml {args} \
    output_dir="{TRYON_OUT}" \
    train.warp_checkpoint="{WARP_OUT}/warp.pt" \
    train.steps=15000 train.batch_size=8 train.num_workers=2 \
    train.sample_every=1000 train.save_every=1000

In [ ]:
sheets = sorted((TRYON_OUT / 'samples').glob('*.png'))
display(Image(str(sheets[-1])))

## 7. Optional — adversarial sharpening pass

Only run this once stage 2 has converged and the output is structurally correct
but soft. It writes to a **separate** output folder so a GAN collapse cannot
destroy the checkpoint you already have.

In [ ]:
GAN_OUT = OUTPUT_ROOT / 'tryon_gan'

# Seed the GAN run from the converged checkpoint.
import shutil
GAN_OUT.mkdir(parents=True, exist_ok=True)
if not (GAN_OUT / 'tryon.pt').exists():
    shutil.copy(TRYON_OUT / 'tryon.pt', GAN_OUT / 'tryon.pt')

!python train_tryon.py --config configs/tryon.yaml {args} \
    output_dir="{GAN_OUT}" \
    train.warp_checkpoint="{WARP_OUT}/warp.pt" \
    loss.gan=0.5 train.gan_start_step=0 train.lr=0.00005 \
    train.steps=19000 train.batch_size=8 train.num_workers=2 \
    train.sample_every=500 train.save_every=500

## 8. Inference — every garment on every person

This is the honest evaluation. Training is self-paired, so reconstructing someone
in their own clothes proves nothing. **Rows are people, columns are garments —
judge the off-diagonal cells.**

In [ ]:
GRID = OUTPUT_ROOT / 'grid.png'

!python infer.py --checkpoint "{TRYON_OUT}/tryon.pt" --grid \
    --root "{DATA_ROOT}" --limit 5 --out "{GRID}"

display(Image(str(GRID)))

## Notes

* **Disconnected?** Re-run from the top. Both training cells resume from the last
  checkpoint in Drive automatically. To start clean instead, add
  `train.resume=false` or delete the output folder.
* **Want to train longer?** Just raise `train.steps` and re-run the cell — the
  schedule is replayed against the new total, so extending a finished run works.
* **Out of memory?** Lower `train.batch_size` to 4, or add `train.accumulate=2`
  to keep the effective batch size while halving memory.
* **Both stages must use the same resolution.** Stage 2 checks this and refuses
  to start on a mismatch, because the TPS kernel is precomputed per resolution.
* Full tuning and diagnostics tables are in the repo `README.md`.